In [ ]:
!pip install prophet

In [ ]:
!unzip dataset_d3_filtrado.zip

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import joblib
import os
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. CARGA Y LIMPIEZA DE DATOS
df = pd.read_csv("dataset_d3_filtrado.csv")
df['datetime'] = pd.to_datetime(df['datetime'])
df = df[df['num_placas'] > 0]
df['produccion_por_placa'] = df['produccion_kWh'] / df['num_placas']
df['nubosidad'] = df['nubosidad'].fillna(0)
df.sort_values(by='datetime', inplace=True)
df['produccion_dia_anterior'] = df.groupby('id_casa')['produccion_por_placa'].shift(24).fillna(0)

# Agrupado final
df_grouped = df.groupby(['id_casa', 'datetime', 'nubosidad', 'festivo', 'produccion_dia_anterior'], as_index=False).agg({'produccion_por_placa': 'mean'})

# 2. ENTRENAMIENTO Y PREDICCIÓN
s.makedirs("modelos_prophet", exist_ok=True)
os.makedirs("resultados_cross_validation", exist_ok=True)
resultados_por_casa = []
errores = []

ids_casas = df_grouped['id_casa'].unique()

for casa_id in ids_casas:
    print(f"Procesando casa {casa_id}...")
    df_casa = df_grouped[df_grouped['id_casa'] == casa_id].copy()
    df_prophet = df_casa.rename(columns={'datetime': 'ds', 'produccion_por_placa': 'y', 'festivo': 'holiday'})

    # Festivos
    festivos = df_prophet[df_prophet['holiday'] == 1][['ds']].copy()
    festivos['holiday'] = 'festivo'

    modelo = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode='additive',
        changepoint_prior_scale=0.1,
        seasonality_prior_scale=10,
        interval_width=0.90,
        holidays=festivos
    )

    modelo.add_regressor('nubosidad')
    modelo.add_regressor('produccion_dia_anterior')

    try:
        modelo.fit(df_prophet)

        # Guardar modelo
        joblib.dump(modelo, f"modelos_prophet/modelo_casa_{casa_id}.pkl")

        # Validación cruzada (OPTIMIZADA)
        df_cv = cross_validation(
            modelo,
            initial='2160 hours',  # 3 meses si tienes varios años de datos
            period='480 hours',    # cada 20 días (más separados)
            horizon='48 hours',    # predicción de 2 días
            parallel='processes'   # MULTIPROCESO
        )

        df_metrics = performance_metrics(df_cv)
        df_metrics['id_casa'] = casa_id
        df_metrics.to_csv(f"resultados_cross_validation/casa_{casa_id}_metrics.csv", index=False)

        # Forecast para análisis posterior
        future = modelo.make_future_dataframe(periods=48, freq='h')
        future = pd.merge(future, df_casa[['datetime', 'nubosidad', 'produccion_dia_anterior']], left_on='ds', right_on='datetime', how='left').drop(columns=['datetime'])
        future.fillna(0, inplace=True)
        forecast = modelo.predict(future)

        resultados_por_casa.append({
            'id_casa': casa_id,
            'forecast': forecast,
            'yhat_mean': forecast['yhat'].mean(),
            'yhat_std': forecast['yhat'].std()
        })

        print(f"Casa {casa_id} completada.")

    except Exception as e:
        print(f"Error en casa {casa_id}: {e}")
        errores.append((casa_id, str(e)))
        continue

# 3. ANÁLISIS FINAL
df_resumen = pd.DataFrame(resultados_por_casa)[['id_casa', 'yhat_mean', 'yhat_std']]
df_resumen.rename(columns={'yhat_mean': 'produccion_media', 'yhat_std': 'variabilidad'}, inplace=True)
df_resumen.to_csv("resumen_predicciones_por_casa.csv", index=False)

print("Todos los modelos entrenados y métricas exportadas.")
if errores:
    print(f"Errores en {len(errores)} casas: {[e[0] for e in errores]}")
